In [1]:
!unzip -q /content/funko-ml-main.zip -d /content/

In [2]:
import os

print("Project files:")
print(os.listdir('/content/funko-ml-main'))

print("\nData files:")
print(os.listdir('/content/funko-ml-main/data'))

print("\nSource files:")
print(os.listdir('/content/funko-ml-main/src'))

Project files:
['.gitignore', 'results', 'slides', 'README.md', 'requirements.txt', 'src', 'notebooks', 'data', 'writeup']

Data files:
['processed.npz', 'processed_table.csv', 'data_curated.csv', 'folds.pkl']

Source files:
['__init__.py', 'evaluate.py']


In [3]:
import sys
import os
import importlib.util

evaluate_path = '/content/funko-ml-main/src/evaluate.py'

spec = importlib.util.spec_from_file_location(
    "evaluate",
    evaluate_path
)

evaluate = importlib.util.module_from_spec(spec)

sys.modules["evaluate"] = evaluate

spec.loader.exec_module(evaluate)

from evaluate import load_data, evaluate_model, save_results

print("Evaluation functions imported successfully!")

Evaluation functions imported successfully!


In [4]:
d, folds = load_data(
    data_dir='/content/funko-ml-main/data'
)

print("Data loaded successfully!")

print("\nAvailable data:")
print(d.keys())

print("\nX_onehot shape:", d['X_onehot'].shape)
print("X_ordinal shape:", d['X_ordinal'].shape)
print("y shape:", d['y'].shape)

print("\nNumber of folds:", len(folds))

Data loaded successfully!

Available data:
dict_keys(['X_onehot', 'X_ordinal', 'y', 'y0', 'onehot_names', 'ordinal_names'])

X_onehot shape: (359, 59)
X_ordinal shape: (359, 4)
y shape: (359,)

Number of folds: 4


In [5]:
from sklearn.tree import DecisionTreeClassifier

print("Decision Tree imported successfully!")

Decision Tree imported successfully!


In [6]:
import numpy as np

print("Target classes:")
print(np.unique(d['y']))

print("\nClass counts:")
print(np.bincount(d['y'])[1:])

Target classes:
[1 2 3 4 5]

Class counts:
[146 153  46  10   4]


In [7]:
res_tree = evaluate_model(
    lambda: DecisionTreeClassifier(),
    d['X_onehot'],
    d['y'],
    folds
)

print("Decision Tree evaluation completed!")

Decision Tree evaluation completed!


In [8]:
print("DECISION TREE RESULTS")
print("=====================")

print("\nTRAIN")
print(f"Precision: {res_tree['train']['precision']:.4f}")
print(f"Recall:    {res_tree['train']['recall']:.4f}")
print(f"F1 Score:  {res_tree['train']['f1']:.4f}")

print("\nVALIDATION")
print(f"Precision: {res_tree['valid']['precision']:.4f}")
print(f"Recall:    {res_tree['valid']['recall']:.4f}")
print(f"F1 Score:  {res_tree['valid']['f1']:.4f}")

print("\nTRAIN CLASS PRECISION")
print(res_tree['train_class_precision'])

print("\nVALIDATION CLASS PRECISION")
print(res_tree['valid_class_precision'])

DECISION TREE RESULTS

TRAIN
Precision: 0.9840
Recall:    0.9833
F1 Score:  0.9833

VALIDATION
Precision: 0.4018
Recall:    0.4011
F1 Score:  0.3994

TRAIN CLASS PRECISION
[0.96056175 1.         1.         1.         1.        ]

VALIDATION CLASS PRECISION
[0.48165878 0.43488649 0.13819444 0.         0.25      ]


In [9]:
print("DECISION TREE: OUR RESULTS vs REFERENCE")
print("========================================")

print("\n                    Our Result     Reference")

print(
    f"Train Precision     {res_tree['train']['precision']:.3f}"
    f"          0.984"
)

print(
    f"Train Recall        {res_tree['train']['recall']:.3f}"
    f"          0.983"
)

print(
    f"Train F1            {res_tree['train']['f1']:.3f}"
    f"          0.983"
)

print(
    f"Valid Precision     {res_tree['valid']['precision']:.3f}"
    f"          0.408"
)

print(
    f"Valid Recall        {res_tree['valid']['recall']:.3f}"
    f"          0.415"
)

print(
    f"Valid F1            {res_tree['valid']['f1']:.3f}"
    f"          0.409"
)

DECISION TREE: OUR RESULTS vs REFERENCE

                    Our Result     Reference
Train Precision     0.984          0.984
Train Recall        0.983          0.983
Train F1            0.983          0.983
Valid Precision     0.402          0.408
Valid Recall        0.401          0.415
Valid F1            0.399          0.409


In [10]:
paper_train_f1 = 0.983
paper_valid_f1 = 0.409

our_train_f1 = res_tree['train']['f1']
our_valid_f1 = res_tree['valid']['f1']

print("F1 SCORE DIFFERENCE")
print("===================")

print(
    f"Train F1 difference: "
    f"{our_train_f1 - paper_train_f1:+.4f}"
)

print(
    f"Validation F1 difference: "
    f"{our_valid_f1 - paper_valid_f1:+.4f}"
)

F1 SCORE DIFFERENCE
Train F1 difference: +0.0003
Validation F1 difference: -0.0096


In [11]:
save_results(
    'decision_tree',
    res_tree
)

print("Decision Tree results saved successfully!")

Decision Tree results saved successfully!


In [12]:
import pandas as pd

result_path = '/content/funko-ml-main/results/decision_tree.csv'

result_df = pd.read_csv(result_path)

print("Saved result:")
display(result_df)

Saved result:


,model,train_precision,train_recall,train_f1,train_bin1_precision,train_bin2_precision,train_bin3_precision,train_bin4_precision,train_bin5_precision,valid_precision,valid_recall,valid_f1,valid_bin1_precision,valid_bin2_precision,valid_bin3_precision,valid_bin4_precision,valid_bin5_precision
0,decision_tree,0.984,0.9833,0.9833,0.9606,1.0,1.0,1.0,1.0,0.4018,0.4011,0.3994,0.4817,0.4349,0.1382,0.0,0.25


In [13]:
print("======================================")
print("DECISION TREE NOTEBOOK COMPLETED")
print("======================================")

print("\nModel: Vanilla Decision Tree")
print("Features: One-Hot Encoded")
print("Cross-validation: 4-fold Stratified")
print("Evaluation: Weighted Precision, Recall, F1")

print("\nOur Results:")
print(
    f"Train F1: {res_tree['train']['f1']:.4f}"
)

print(
    f"Validation F1: {res_tree['valid']['f1']:.4f}"
)

print("\nSaved:")
print("results/decision_tree.csv")
print("======================================")

DECISION TREE NOTEBOOK COMPLETED

Model: Vanilla Decision Tree
Features: One-Hot Encoded
Cross-validation: 4-fold Stratified
Evaluation: Weighted Precision, Recall, F1

Our Results:
Train F1: 0.9833
Validation F1: 0.3994

Saved:
results/decision_tree.csv
